# Scaling-in Payoff Model

This notebook reproduces the static comparison in Section 3.3.

There are three entry policies when price first reaches `L1` and may later reach the lower level `L2` before reverting to `F`:

- all-in at `L1`;
- wait and all-in at `L2`;
- average-in one contract at each level.

Under the fixed-probability model, average-in is never the best expected-profit choice.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


In [ ]:
F = 110.0
L1 = 100.0
L2 = 95.0
p_hat = (F - L1) / (F - L2)

p_grid = np.linspace(0, 1, 101)
profits = pd.DataFrame(
    {
        "p": p_grid,
        "all_in_L1": 2 * (F - L1),
        "all_in_L2": 2 * p_grid * (F - L2),
        "average_in": (F - L1) + p_grid * (F - L2),
    }
).set_index("p")
profits["best_method"] = profits.idxmax(axis=1)
profits.head()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
profits[["all_in_L1", "all_in_L2", "average_in"]].plot(ax=ax, linewidth=2)
ax.axvline(p_hat, color="black", linestyle="--", linewidth=1, label=f"transition p={p_hat:.2f}")
ax.set_title("Expected profit under fixed probability p")
ax.set_xlabel("probability of first dropping from L1 to L2")
ax.set_ylabel("expected profit points")
ax.legend()
plt.tight_layout();


In [ ]:
margin_vs_best = profits[["all_in_L1", "all_in_L2"]].max(axis=1) - profits["average_in"]

pd.Series(
    {
        "F": F,
        "L1": L1,
        "L2": L2,
        "transition_probability": p_hat,
        "minimum_margin_by_which_average_in_loses": margin_vs_best.min(),
        "average_in_is_ever_best": bool((profits["best_method"] == "average_in").any()),
    }
)


In [ ]:
rng = np.random.default_rng(7)
trials = 2_000
p_path = np.clip(rng.normal(loc=0.55, scale=0.25, size=trials), 0, 1)
realized_drop = rng.random(trials) < p_path

all_in_L1 = np.full(trials, 2 * (F - L1))
all_in_L2 = np.where(realized_drop, 2 * (F - L2), 0.0)
average_in = np.where(realized_drop, (F - L1) + (F - L2), F - L1)

simulation = pd.DataFrame(
    {
        "all_in_L1": all_in_L1,
        "all_in_L2": all_in_L2,
        "average_in": average_in,
    }
)
simulation.agg(["mean", "std"])


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
simulation.cumsum().plot(ax=ax, linewidth=2)
ax.set_title("One simulated out-of-sample path with changing probabilities")
ax.set_xlabel("trade number")
ax.set_ylabel("cumulative profit points")
plt.tight_layout();


## Interpretation

In the fixed-probability arithmetic model, average-in lies between the two all-in alternatives and never dominates both. The live-trading argument for scaling-in must therefore come from changing conditions, smoother realized risk, market impact, or robustness rather than from this static expected-profit comparison.
